# Thai speech pipeline: VAD → ASR → speaker diarization

One recording through all three steps of the manual. Each part is the code of its own notebook (`vad.ipynb`,
`ASR_transcription.ipynb`, `diarization.ipynb`), changed only where the parts meet.

| Part | Reads | Writes, next to the recording |
|---|---|---|
| 1. Voice activity detection | the recording | `segments.csv`, `segments.txt`, `segments/segment_0000.wav`… |
| 2. Speech-to-text | the segment WAVs | `hyp_<model>.jsonl`, `results.jsonl`, `results.segments.csv` |
| 3. Speaker diarization | the recording, and Parts 1 and 2 | `<name>.community-1.rttm`, `<name>.who-said-what.csv` |

Part 3 runs diarization once on the whole recording, not on each segment: each run names its own speakers, so
`SPEAKER_00` in one segment would not be `SPEAKER_00` in the next. Each segment then gets the speaker with the most
time inside it.

**Run it**

- **Locally:** `uv sync`, put `HF_TOKEN=hf_...` in `.env` (see `.env.example`), then `uv run jupyter lab`. Part 2
  transcribes with three Whisper large-v3 models and needs a GPU: locally it only transcribes segments missing from
  `results.jsonl`, so make that file in Colab first (`data/clean/` has one).
- **Colab:** *Runtime → Change runtime type → T4 GPU*, add the token as a secret named `HF_TOKEN`, upload `data/`
  with the Files panel, run 0.1, then *Runtime → Restart session* and *Run all*.
- Part 3 needs a Hugging Face token, with the conditions of
  [`pyannote/speaker-diarization-community-1`](https://huggingface.co/pyannote/speaker-diarization-community-1)
  accepted once for your account. Never paste the token into a cell.

The expected outputs of Parts 0 to 3 come from one run on a Mac (CPU) on 2026-11-19 with `data/clean/clean.wav`;
Part 2 transcribed nothing there, because `results.jsonl` (from a Colab T4 run) had every segment. The extras after
Part 3 keep the diarization notebook's outputs (Colab T4, 2026-11-11). On another machine the numbers can move a little.

# Part 0. Setup

## 0.1 Install (Colab only)

Locally, `uv sync` installed all of this, and the cell does nothing. In Colab it installs the versions in this
repository's `uv.lock` and stops: *Runtime → Restart session*, then *Run all* (it then sees pyannote and skips).

pip prints a red `ERROR` with a list of dependency conflicts: Colab's own Google Cloud, TensorFlow and ADK packages
want a newer `protobuf` and `opentelemetry` than pyannote.audio 4.0.0 allows. This notebook uses none of them.

In [ ]:
# 0.1 install what Colab lacks (Colab only)
import importlib.util
import sys

if "google.colab" in sys.modules and importlib.util.find_spec("pyannote.audio") is None:
    !pip install -q silero-vad==6.2.2 onnxruntime ten-vad==1.0.6.9 webrtcvad-wheels==2.0.14.post1 pythainlp==5.4.0 jiwer==4.0.0 pyannote.audio==4.0.0 torch==2.9.1 torchaudio==2.9.1 torchvision==0.24.1 torchcodec==0.9.1 pyannote.metrics==4.1
    raise SystemExit("installed: Runtime → Restart session, then Run all")

## 0.2 Settings, versions, device and token

`AUDIO_PATH` is the only path to change for your own recording. `VAD_MODEL` picks the one VAD Part 1 runs.

In [ ]:
# 0.2 settings, versions, device, token
import gc
import glob
import json
import os
import wave
from importlib.metadata import version

os.environ["PYANNOTE_METRICS_ENABLED"] = "0"  # pyannote.audio's usage telemetry off
import numpy as np
import torch

AUDIO_PATH = "data/clean/clean.wav"  # your recording: 16 kHz mono 16-bit WAV, or convert it in 1.1
AUDIO_DIR = os.path.dirname(AUDIO_PATH)  # every output goes here
VAD_MODEL = "silero"  # "silero", "ten" or "webrtc"

IN_COLAB = "google.colab" in sys.modules
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def load_dotenv(path=".env"):
    """KEY=value lines from .env; variables already set win."""
    if os.path.exists(path):
        for line in open(path):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip().strip("\"'"))


if IN_COLAB:  # the secret named HF_TOKEN
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
else:  # .env, or the environment Jupyter started in
    load_dotenv()
    HF_TOKEN = os.environ.get("HF_TOKEN")

assert np.__version__ == version("numpy"), "numpy changed on disk, restart the session"
for pkg in ["torch", "silero-vad", "pythainlp", "jiwer", "pyannote.audio", "pyannote.metrics"]:
    print(pkg, version(pkg))
print("device:", DEVICE, "| HF_TOKEN set:", bool(HF_TOKEN))

Expected output:

```
(fill in from a run)
```

`device: cpu` is expected on a Mac; in Colab it means the runtime has no GPU. Part 3 stops if `HF_TOKEN set` is
`False`. The numpy check catches Colab keeping the old numpy after 0.1: restart the session.

# Part 1. Voice activity detection

Finds the stretches of speech. Part 2 transcribes each one and Part 3 gives each one a speaker.

## 1.1 Convert your audio to 16 kHz mono WAV

Skipping this step is risky: a wrong sample rate (e.g. 44.1 kHz) makes Silero and WebRTC raise an error, but TEN VAD
runs without one, and so does stereo audio in every model. Both give wrong results. Set `CONVERT_FROM` to convert
another file (any format FFmpeg reads) into `AUDIO_PATH`.

In [ ]:
# 1.1 convert to 16 kHz mono 16-bit WAV, then check
CONVERT_FROM = None  # e.g. "data/input.mp3"; None: AUDIO_PATH already is one

if CONVERT_FROM:
    !ffmpeg -y -i "{CONVERT_FROM}" -ac 1 -ar 16000 "{AUDIO_PATH}"
    # -ac 1 makes it mono, -ar 16000 sets the sample rate to 16 kHz; the output is 16-bit PCM WAV

with wave.open(AUDIO_PATH, "rb") as f:
    assert (f.getframerate(), f.getnchannels(), f.getsampwidth()) == (16000, 1, 2), "not 16 kHz mono 16-bit: set CONVERT_FROM"
    print(AUDIO_PATH, f"{f.getnframes() / f.getframerate():.1f} s, 16 kHz mono 16-bit")

Expected output:

```
(fill in from a run)
```

## 1.2 Find the speech segments

Three models, each in its own function; only `VAD_MODEL` (set in 0.2) runs. Each returns the same `(start, end)`
pairs in seconds.

### Silero

In [ ]:
# 1.2a Silero
INT16_SCALE = 2**15  # 16-bit samples range -32768..32767; dividing gives -1.0..1.0, the range Silero expects


def silero_segments(path):
    from silero_vad import load_silero_vad, get_speech_timestamps

    with wave.open(path, "rb") as f:
        sample_rate = f.getframerate()
        pcm = f.readframes(f.getnframes())
    wav = torch.from_numpy(np.frombuffer(pcm, dtype=np.int16) / INT16_SCALE).float()
    model = load_silero_vad()
    segs = get_speech_timestamps(wav, model, sampling_rate=sample_rate, return_seconds=True)  # default threshold is 0.5; see Tuning
    return [(seg["start"], seg["end"]) for seg in segs]

### TEN VAD

In [ ]:
# 1.2b TEN VAD
HOP = 256  # samples per frame: 256 (16 ms) or 160 (10 ms)


def ten_segments(path):
    from ten_vad import TenVad

    with wave.open(path, "rb") as f:
        sample_rate = f.getframerate()
        pcm = np.frombuffer(f.readframes(f.getnframes()), dtype=np.int16)
    vad = TenVad(HOP, 0.5)  # threshold 0.5; see Tuning
    segs, start = [], None
    for i in range(0, len(pcm) - HOP + 1, HOP):
        t = i / sample_rate
        _, is_speech = vad.process(pcm[i:i + HOP])
        if is_speech:
            if start is None:
                start = t
        elif start is not None:
            segs.append((start, t))
            start = None
    if start is not None:
        segs.append((start, len(pcm) / sample_rate))
    return segs

### WebRTC

In [ ]:
# 1.2c WebRTC
FRAME_MS = 30  # must be 10, 20 or 30


def webrtc_segments(path):
    import webrtcvad

    vad = webrtcvad.Vad(3)  # mode 0 to 3
    with wave.open(path, "rb") as f:
        sample_rate = f.getframerate()
        pcm = f.readframes(f.getnframes())
    frame_bytes = int(sample_rate * FRAME_MS / 1000) * 2  # 2 bytes per 16-bit sample
    segs, start = [], None
    for i in range(0, len(pcm) - frame_bytes + 1, frame_bytes):
        t = i / 2 / sample_rate
        if vad.is_speech(pcm[i:i + frame_bytes], sample_rate):
            if start is None:
                start = t
        elif start is not None:
            segs.append((start, t))
            start = None
    if start is not None:
        segs.append((start, len(pcm) / 2 / sample_rate))
    return segs

### Run the chosen model

In [ ]:
# 1.2d run VAD_MODEL
segments = {"silero": silero_segments, "ten": ten_segments, "webrtc": webrtc_segments}[VAD_MODEL](AUDIO_PATH)
print(f"{VAD_MODEL}: {len(segments)} segments, {sum(e - s for s, e in segments):.1f} s of speech")
for s, e in segments[:3]:
    print(f"{s:.2f}s -> {e:.2f}s")

Expected output:

```
(fill in from a run)
```

## 1.3 Check the output by ear

The scores in this manual come from our recordings, not yours, so listen to a sample of your own output before relying on it. This check is our advice, not something we measured.

a. Save the segments next to the audio, twice: `segments.csv`, one row per segment with the audio path, start, end and a label; and `segments.txt`, a label file that Audacity (free, audacityteam.org) can open, with start, end and a label separated by tabs.

In [ ]:
import csv
import os

# same columns as the reference.csv files in data/
with open(os.path.join(AUDIO_DIR, "segments.csv"), "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["audio_path", "start_sec", "end_sec", "label"])
    writer.writerows((AUDIO_PATH, f"{s:.3f}", f"{e:.3f}", "speech") for s, e in segments)

# Audacity label file: start, end, label, separated by tabs, no header
with open(os.path.join(AUDIO_DIR, "segments.txt"), "w") as f:
    for s, e in segments:
        f.write(f"{s:.3f}\t{e:.3f}\tspeech\n")

b. In Audacity, open the WAV, then choose File > Import > Labels and pick `segments.txt` from the audio's folder. Each segment shows as a label under the waveform.

c. Play labels picked at random, and gaps between labels. Is each label speech, starting and ending with the words? Is each gap free of speech?

## 1.4 Save segments to segment WAVs

In [ ]:
# 1.4 one WAV per segment, segment_0000.wav, segment_0001.wav… in file order: Part 2 transcribes these
with wave.open(AUDIO_PATH, "rb") as f:
    sample_rate = f.getframerate()
    pcm = f.readframes(f.getnframes())

os.makedirs(os.path.join(AUDIO_DIR, "segments"), exist_ok=True)
for i, (start, end) in enumerate(segments):
    first, last = int(start * sample_rate) * 2, int(end * sample_rate) * 2  # 2 bytes per 16-bit sample
    with wave.open(os.path.join(AUDIO_DIR, "segments", f"segment_{i:04d}.wav"), "wb") as out:
        out.setnchannels(1)
        out.setsampwidth(2)
        out.setframerate(sample_rate)
        out.writeframes(pcm[first:last])
print(len(segments), "WAVs in", os.path.join(AUDIO_DIR, "segments"))

## 1.5 Optional: evaluate against a reference

Skip this step unless you have a reference: speech segments a person labeled by hand, saved as `reference.csv` in the audio's folder, with columns `start_sec` and `end_sec` (a `label` column is optional; only `speech` rows count). The sample files in `data/` come with one.

It scores `segments`, so it scores `VAD_MODEL`. The metrics are the manual's:

| Metric | Meaning |
|---|---|
| FN rate | Share of reference speech the model missed: FN / (TP + FN) |
| FP rate | Share of reference non-speech the model called speech: FP / (FP + TN) |
| DCF | 0.75 × FN rate + 0.25 × FP rate; lower is better |

A 200 ms collar around each reference boundary (100 ms on each side) is not scored, because hand-placed boundaries are only accurate to a few frames. Scoring uses pyannote.metrics, as the manual's numbers do. It is already in this project's environment; elsewhere, run `uv pip install pyannote.metrics`.

Without a reference the cell says so and does nothing, so *Run all* goes on.

In [ ]:
# 1.5 score the segments against a hand-labelled reference.csv, if there is one
import csv
from pyannote.core import Annotation, Segment, Timeline
from pyannote.metrics.detection import DetectionCostFunction

REFERENCE_CSV = os.path.join(AUDIO_DIR, "reference.csv")  # columns start_sec, end_sec; a label column is optional
COLLAR = 0.2  # seconds around each reference boundary that are not scored: 100 ms on each side


def to_annotation(segs):
    """Turn (start, end) pairs into the speech annotation pyannote.metrics scores."""
    annotation = Annotation()
    for s, e in segs:
        annotation[Segment(s, e)] = "speech"
    return annotation


if not os.path.exists(REFERENCE_CSV):
    print(f"no reference at {REFERENCE_CSV}: skipped")
else:
    with open(REFERENCE_CSV, newline="") as f:
        reference_segments = [
            (float(row["start_sec"]), float(row["end_sec"]))
            for row in csv.DictReader(f)
            if row.get("label", "speech") == "speech"
        ]
    with wave.open(AUDIO_PATH, "rb") as f:
        audio_seconds = f.getnframes() / f.getframerate()

    # DCF = 0.75 x FN rate + 0.25 x FP rate, as NIST defines it
    metric = DetectionCostFunction(collar=COLLAR)
    uem = Timeline([Segment(0, audio_seconds)])  # score the whole file, so silence at either end counts as non-speech
    c = metric(to_annotation(reference_segments), to_annotation(segments), uem=uem, detailed=True)
    fn_rate = c["miss"] / c["positive class total"]           # share of speech missed
    fp_rate = c["false alarm"] / c["negative class total"]    # share of non-speech called speech
    print(f"{VAD_MODEL}: FN rate {fn_rate:.3f}, FP rate {fp_rate:.3f}, DCF {c[metric.name]:.3f}")

Expected output:

```
(fill in from a run)
```

# Part 2. Speech-to-text: three models, pairwise agreement and best text

```
Part 1's segment WAVs
   ├── distill  ─► hypothesis text ─┐
   ├── typhoon  ─► hypothesis text ─┼─► normalization (+ PyThaiNLP word tokenization for WER_norm)
   └── pathumma ─► hypothesis text ─┘        │
                                             ▼
        compare every pair: CER_raw, CER_norm, WER_norm
          distill <-> pathumma, distill <-> typhoon, pathumma <-> typhoon
                                             │
                                             ▼
        Mean_Score(A) = (CER_norm(A,B) + CER_norm(A,C)) / 2   ─►  lowest score  ─►  best text
```

There is no reference transcript: the three models are scored against each other. A model that agrees with the
other two has a low mean score, so its transcript is taken as the best text.

**Output**: `results.jsonl` next to the recording, one line per segment. Transcription (2.2) needs a GPU; every step
skips segments that are already done, so if Colab disconnects, *Run all* again. Steps 2.3 to 2.6 need no GPU.

## 2.1 Config

The segments are Part 1's, by number: `segment_0000.wav` is Part 1's first segment. `results.segments.csv` remembers
which times those numbers had when the transcripts were made, so a changed Part 1 cannot silently pair new segments
with old text.

In [ ]:
# 2.1 config: the models, the segments from Part 1, the output files
from tqdm.auto import tqdm

MODELS = {
    "distill":  "biodatlab/distill-whisper-th-large-v3",
    "pathumma": "nectec/Pathumma-whisper-th-large-v3",
    "typhoon":  "typhoon-ai/typhoon-whisper-large-v3",
}
BATCH_SIZE = 16
SR         = 16000
MAX_SEC    = 30.0     # longer segments are transcribed one at a time in long-form mode
DTYPE      = torch.float16 if DEVICE == "cuda" else torch.float32

SEG_DIR = os.path.join(AUDIO_DIR, "segments")
SEGMENTS = {f"segment_{i:04d}.wav": os.path.join(SEG_DIR, f"segment_{i:04d}.wav") for i in range(len(segments))}
SEG_TIMES = dict(zip(SEGMENTS, segments))  # file name -> (start, end) in the recording

HYP_JSONL     = {m: os.path.join(AUDIO_DIR, f"hyp_{m}.jsonl") for m in MODELS}  # one model's transcripts, appended batch by batch
RESULTS_JSONL = os.path.join(AUDIO_DIR, "results.jsonl")                         # final output, one line per segment
SNAPSHOT      = os.path.join(AUDIO_DIR, "results.segments.csv")                  # the segment times the transcripts belong to


def read_jsonl(path):
    if not os.path.exists(path): return []
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def append_jsonl(path, rows):
    with open(path, "a", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")


lines = ["segment,start_sec,end_sec"] + [f"{a},{s:.3f},{e:.3f}" for a, (s, e) in SEG_TIMES.items()]
if os.path.exists(SNAPSHOT):
    assert open(SNAPSHOT).read().splitlines() == lines, (
        "Part 1's segments changed since these transcripts were made: delete results.jsonl, hyp_*.jsonl and "
        f"results.segments.csv in {AUDIO_DIR}, then run Part 2 again")
else:
    with open(SNAPSHOT, "w") as f:
        f.write("\n".join(lines) + "\n")

in_results = {r["audio_path"] for r in read_jsonl(RESULTS_JSONL)}
print(f"{len(SEGMENTS)} segments, {len(in_results & set(SEGMENTS))} already in results.jsonl,"
      f" {len(set(SEGMENTS) - in_results)} to transcribe | device: {DEVICE}")

Expected output:

```
(fill in from a run)
```

## 2.2 Transcribe with each model

Same settings for all three models: fp16, greedy decoding, `language="th"`, `task="transcribe"`, one model on the
GPU at a time. Segments of 30 s or less are transcribed in batches. Longer ones are transcribed one at a time in
Whisper's long-form mode, so nothing after 30 s is cut off.

Each model writes `hyp_<model>.jsonl` (`{"audio_path", "hyp"}` or `{"audio_path", "error"}`) after every batch.
A segment that cannot be read or transcribed gets its error message instead of a transcript, like the
`errors` field in the output.

Segments already in `results.jsonl` are skipped too, and `transformers` is only loaded when there is something to
transcribe, so this cell also runs locally once `results.jsonl` is complete.

In [ ]:
# 2.2 transcribe with each model (GPU)
import soundfile as sf


def load_audio(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim > 1: y = y.mean(axis=1)
    if sr != SR:
        import librosa
        y = librosa.resample(y, orig_sr=sr, target_sr=SR)
    return y


def load_model(name):
    from transformers import WhisperForConditionalGeneration, WhisperProcessor
    proc = WhisperProcessor.from_pretrained(MODELS[name])
    model = WhisperForConditionalGeneration.from_pretrained(MODELS[name], torch_dtype=DTYPE).to(DEVICE).eval()
    model.generation_config.forced_decoder_ids = None      # language and task are passed explicitly below
    return model, proc


@torch.inference_mode()
def transcribe(model, proc, waves):
    long = max(len(w) for w in waves) > MAX_SEC * SR
    if long:   # long-form: keep the whole segment, Whisper decodes it 30 s at a time
        feats = proc(waves, sampling_rate=SR, return_tensors="pt", truncation=False, padding="longest", return_attention_mask=True)
    else:
        feats = proc(waves, sampling_rate=SR, return_tensors="pt", return_attention_mask=True)
    ids = model.generate(
        input_features=feats.input_features.to(DEVICE, DTYPE),
        attention_mask=feats.attention_mask.to(DEVICE),
        language="th", task="transcribe", return_timestamps=long, num_beams=1, do_sample=False,
    )
    return [t.strip() for t in proc.tokenizer.batch_decode(ids, skip_special_tokens=True)]


def run_model(name):
    done = {r["audio_path"] for r in read_jsonl(HYP_JSONL[name])} | {r["audio_path"] for r in read_jsonl(RESULTS_JSONL)}
    todo = [a for a in SEGMENTS if a not in done]
    if not todo:
        print(f"{name}: already done"); return
    model, proc = load_model(name)
    lengths = {a: sf.info(SEGMENTS[a]).duration if os.path.exists(SEGMENTS[a]) else 0 for a in todo}
    short = [a for a in todo if lengths[a] <= MAX_SEC]
    batches = [short[i:i + BATCH_SIZE] for i in range(0, len(short), BATCH_SIZE)] + [[a] for a in todo if lengths[a] > MAX_SEC]
    for batch in tqdm(batches, desc=name):
        rows, waves, ok = [], [], []
        for a in batch:
            try: waves.append(load_audio(SEGMENTS[a])); ok.append(a)
            except Exception as e: rows.append({"audio_path": a, "error": str(e)})
        if ok:
            try:
                rows += [{"audio_path": a, "hyp": h} for a, h in zip(ok, transcribe(model, proc, waves))]
            except Exception:                       # find which segment failed: retry one by one
                for a, w in zip(ok, waves):
                    try: rows.append({"audio_path": a, "hyp": transcribe(model, proc, [w])[0]})
                    except Exception as e: rows.append({"audio_path": a, "error": str(e)})
        append_jsonl(HYP_JSONL[name], rows)
    del model; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()


for name in MODELS:
    run_model(name)
print("transcription finished")

Expected output when `results.jsonl` already has every segment:

```
(fill in from a run)
```

## 2.3 Normalization functions (for CER_norm and WER_norm)

Needs no GPU, so this and the next steps can be rerun at any time.

**Normalization.** The three models write the same speech in different styles, such as spaces, ๆ, punctuation
and letter case. These are formatting differences, not recognition errors. Normalization removes them, so the
score measures only real differences in content. The same steps are applied to the reference and to every model's output.

- **Lower case**: English words can be written as TikTok or tiktok, so everything is converted to lower case.
- **Thai character normalization**: `pythainlp.util.normalize` fixes characters typed in a different order or
  duplicated, which look the same but are different strings (`เเ` becomes `แ`, `นํ้า` becomes `น้ำ`).
- **ๆ (maiyamok)**: ๆ means the word before it is spoken twice, so just deleting it would break the match.
  `pythainlp.util.maiyamok` first expands it into the repeated word (`คนๆนก` becomes `คน คน นก`).
- **Number words to digits**: spelled-out numbers become digits, so a model that writes numbers as words
  (typhoon never writes digits) is not penalised. Applied to every model and the reference. It uses PyThaiNLP's
  `thaiword_to_num` on whole `newmm` tokens only:
  - ordinary numbers: `สองพันสิบ` becomes `2010`, `เก้าร้อยแปดสิบจุดเก้าห้า` becomes `980.95`
  - digits read one by one (two or more in a row): `ศูนย์แปดหนึ่งสอง` becomes `0812`
  - words that only contain a number word, such as `ห้าม` or `เก้าอี้`, are left alone
  - a lone `ศูนย์`, `ร้อย`, `พัน`, `หมื่น`, `แสน` or `ล้าน` is left as written (`ศูนย์` usually means centre)
  - known limits: `หนึ่ง` used as "a/one" becomes `1` (on both sides, so it mostly cancels), and mixed forms such
    as `100 ล้าน` against `ร้อยล้าน` still count as different
- **Punctuation**: models write symbols differently, so all of `string.punctuation` plus the Thai symbol ฯ
  are removed (ๆ is already expanded in the step above). The one exception is a decimal point between two digits, so `980.95` does not turn into `98095`.
- **Space**: each model puts spaces in different places, so all whitespace is removed.
- **Word tokenizer** (WER only): Thai has no spaces between words, so the normalised text is split into words
  with PyThaiNLP `newmm`.

In [ ]:
# 2.3 normalization functions
import re, string
from pythainlp.util import maiyamok, thaiword_to_num, normalize as th_normalize
from pythainlp.tokenize import word_tokenize

# --- Normalization: removes formatting differences (case, Thai char order, ๆ, number words, punctuation, spaces) ---
# all punctuation and ฯ, except a decimal point between digits (980.95 stays 980.95).
# ๆ is not listed: expand_maiyamok has already turned it into the repeated word.
SYMBOLS = re.compile(r"(?!(?<=\d)\.(?=\d))[ฯ%s]" % re.escape(string.punctuation))

def remove_space(s):
    return "".join(s.split())

def expand_maiyamok(s):
    "ๆ means the word before it is spoken twice: คนๆนก -> คนคนนก"
    return "".join(maiyamok(s))

def remove_punctuation(s):
    return SYMBOLS.sub("", s)

def to_lower(s):
    return s.lower()

DIGIT_WORDS = {"ศูนย์": "0", "หนึ่ง": "1", "สอง": "2", "สาม": "3", "สี่": "4", "ห้า": "5", "หก": "6", "เจ็ด": "7", "แปด": "8", "เก้า": "9"}
BARE_UNITS  = {"ศูนย์", "ร้อย", "พัน", "หมื่น", "แสน", "ล้าน"}   # left as written when they stand alone

def is_number_word(tok):
    "True only when the whole token is a Thai number word, so ห้าม or เก้าอี้ are never touched"
    if tok in DIGIT_WORDS: return True
    if "ลบ" in tok: return False                                 # ลบ is more often "delete" than "minus"
    try: thaiword_to_num(tok); return True
    except Exception: return False

def convert_number_run(run):
    "run = consecutive number-word tokens, possibly ending in จุด plus decimal digits"
    if len(run) == 1 and run[0] in BARE_UNITS:                   # ศูนย์ = centre, หลายร้อย, แสนสาหัส, 3.5 ล้าน
        return run[0]
    if len(run) >= 2 and all(t in DIGIT_WORDS for t in run):     # read digit by digit: phone numbers, codes
        return "".join(DIGIT_WORDS[t] for t in run)
    try:
        if "จุด" in run:
            k = run.index("จุด")
            frac = "".join(DIGIT_WORDS[t] for t in run[k + 1:])  # decimals are read digit by digit
            return f"{thaiword_to_num(''.join(run[:k]))}.{frac}"
        return str(thaiword_to_num("".join(run)))
    except Exception:
        return "".join(run)                                      # not a well-formed number: leave as written

def numbers_to_digits(s):
    toks, out, i = word_tokenize(s, engine="newmm"), [], 0
    while i < len(toks):
        j, decimal = i, False
        while j < len(toks):
            t = toks[j]
            if decimal: ok = t in DIGIT_WORDS                    # after จุด only single digits belong to the number
            elif t == "จุด": ok = decimal = j > i and j + 1 < len(toks) and toks[j + 1] in DIGIT_WORDS
            else: ok = is_number_word(t)
            if not ok: break
            j += 1
        if j > i: out.append(convert_number_run(toks[i:j])); i = j
        else: out.append(toks[i]); i += 1
    return "".join(out)

def normalize_text(s):
    s = to_lower(str(s))
    s = th_normalize(s)           # fix Thai character order and duplicated vowels/tone marks (เเ -> แ, นํ้า -> น้ำ)
    s = expand_maiyamok(s)        # before removing symbols, so ๆ becomes the repeated word
    s = numbers_to_digits(s)      # สองพันสิบ -> 2010, so a model that spells numbers out (typhoon) is not penalised
    s = remove_punctuation(s)
    return remove_space(s)

def tokenize(s):
    "Thai has no spaces between words, so split with PyThaiNLP newmm (used for WER only)"
    return [t for t in word_tokenize(s, engine="newmm", keep_whitespace=False) if t.strip()]

## 2.4 Normalize every hypothesis

Loads each model's transcripts and runs the normalization once per transcript:
- `norm[model][audio_path]`: normalized text, used for CER_norm
- `words[model][audio_path]`: the normalized text split into PyThaiNLP `newmm` words, used for WER_norm

Only segments not yet in `results.jsonl` are processed, so a rerun skips finished work.

In [ ]:
# 2.4 normalize every hypothesis
# load every model's transcripts
hyps = {m: {} for m in MODELS}     # hyps[model][audio_path] = transcript as written
errs = {m: {} for m in MODELS}     # errs[model][audio_path] = error message
for m in MODELS:
    for r in read_jsonl(HYP_JSONL[m]):
        if "hyp" in r: hyps[m][r["audio_path"]] = r["hyp"]; errs[m].pop(r["audio_path"], None)
        else:          errs[m][r["audio_path"]] = r["error"]; hyps[m].pop(r["audio_path"], None)

# segments still to score: every model has finished them and they are not in results.jsonl yet
done = {r["audio_path"] for r in read_jsonl(RESULTS_JSONL)}
todo = [a for a in SEGMENTS if a not in done and all(a in hyps[m] or a in errs[m] for m in MODELS)]
print(f"{len(done)} segments already in results.jsonl, {len(todo)} to normalize and score")

# normalize every transcript once (for CER_norm) and split it into words (for WER_norm)
norm  = {m: {} for m in MODELS}    # norm[model][audio_path]  = normalized text
words = {m: {} for m in MODELS}    # words[model][audio_path] = newmm word list of the normalized text
for m in MODELS:
    for a in tqdm([a for a in todo if a in hyps[m]], desc=f"normalize {m}"):
        norm[m][a] = normalize_text(hyps[m][a])
        words[m][a] = tokenize(norm[m][a])

# check one segment
if todo:
    a = next((a for a in todo if all(a in hyps[m] for m in MODELS)), todo[0])
    print("\n", a)
    for m in MODELS:
        if a in hyps[m]: print(f"  {m:9s} raw : {hyps[m][a]}\n  {'':9s} norm: {norm[m][a]}\n  {'':9s} words: {words[m][a]}")

Expected output when `results.jsonl` already has every segment:

```
(fill in from a run)
```

## 2.5 Score: compare the hypotheses and pick the best text

Uses the normalized text and words from 2.4; nothing is normalized again here.

For every segment, each pair of models is scored. The model that comes first alphabetically is the reference:

| Score | Text used |
|---|---|
| `cer_raw` | the transcripts as written, outer whitespace trimmed |
| `cer_norm` | the normalized transcripts |
| `wer_norm` | the normalized transcripts split into PyThaiNLP `newmm` words |

Then each model gets `Mean_Score(A) = (CER_norm(A,B) + CER_norm(A,C)) / 2` (`mean_agreement_per_model`).
The lowest mean score is the best model; a tie goes to the model first in alphabetical order.
`best_text` (and `text`) is the best model's transcript with all spaces removed. `unanimous` is true when every
`cer_norm` is 0. All scores are rounded to 4 decimals.

If a model fails on a segment, its hypothesis is `null` and its error message goes in `errors`. Fewer than two
transcripts cannot be compared, so then `pairs` is empty, `best_model` and `best_text` are `null` and `text` is `""`.

In [ ]:
# 2.5 compare the hypotheses, pick the best text, write results.jsonl
import itertools
import jiwer

def cer(ref, hyp):
    "character error rate with ref as the reference; jiwer rejects an empty reference, so that case is handled here"
    if not ref: return 0.0 if not hyp else 1.0
    return jiwer.cer(ref, hyp)

def wer(ref_words, hyp_words):
    "ref_words / hyp_words are newmm word lists; jiwer splits words on spaces"
    if not ref_words: return 0.0 if not hyp_words else 1.0
    return jiwer.wer(" ".join(ref_words), " ".join(hyp_words))

def compare(raw, nrm, wrd):
    """raw / nrm / wrd = {model: raw text / normalized text / word list} for the models that succeeded.
    Every pair is scored with the first model (alphabetical) as the reference. Mean_Score(A) = mean of CER_norm
    over the pairs that contain A; with three models that is (CER_norm(A,B) + CER_norm(A,C)) / 2.
    Lowest mean = best model (ties: alphabetical)."""
    names = sorted(raw)
    if len(names) < 2:            # nothing to compare against
        return {"pairs": [], "mean_agreement_per_model": {}, "unanimous": False, "best_model": None, "best_text": None}
    pairs = []
    for a, b in itertools.combinations(names, 2):
        pairs.append({"pair": [a, b],
                      "cer_raw":  round(cer(raw[a].strip(), raw[b].strip()), 4),
                      "cer_norm": round(cer(nrm[a], nrm[b]), 4),
                      "wer_norm": round(wer(wrd[a], wrd[b]), 4)})
    mean = {}
    for m in names:
        own = [p["cer_norm"] for p in pairs if m in p["pair"]]
        mean[m] = round(sum(own) / len(own), 4)
    best = min(names, key=lambda m: mean[m])
    return {"pairs": pairs, "mean_agreement_per_model": mean,
            "unanimous": all(p["cer_norm"] == 0 for p in pairs),
            "best_model": best, "best_text": "".join(raw[best].split())}

def make_record(a):
    ok = [m for m in MODELS if a in hyps[m]]
    comp = compare({m: hyps[m][a] for m in ok}, {m: norm[m][a] for m in ok}, {m: words[m][a] for m in ok})
    return {"audio_path": a,
            "text": comp["best_text"] or "",
            "hypotheses": {m: hyps[m].get(a) for m in sorted(MODELS)},
            "errors": {m: errs[m][a] for m in sorted(MODELS) if a in errs[m]},
            "comparison": comp}

buf = []
for a in tqdm(todo, desc="score"):
    buf.append(make_record(a))
    if len(buf) >= 1000: append_jsonl(RESULTS_JSONL, buf); buf = []
append_jsonl(RESULTS_JSONL, buf)
print("results:", RESULTS_JSONL)

## 2.6 Compare scores (optional)

This step only reads `results.jsonl` and prints summary tables. It writes nothing, so you can skip it and
`results.jsonl` stays the same. It is a quick sanity check: one model winning almost every segment, or a very low
unanimous share, would suggest something went wrong.

In [ ]:
# 2.6 summary of results.jsonl
import pandas as pd

res = read_jsonl(RESULTS_JSONL)
comp = [r["comparison"] for r in res if r["comparison"]["best_model"] is not None]
print(f"{len(res)} segments, {len(comp)} compared, {len(res) - len(comp)} without a best text (errors)")

print("\nhow often each model gives the best text")
display(pd.Series([c["best_model"] for c in comp]).value_counts().rename("segments").to_frame()
          .assign(percent=lambda d: (100 * d.segments / d.segments.sum()).round(1)))

print("\nmean Mean_Score per model (lower = agrees more with the other two)")
display(pd.DataFrame([c["mean_agreement_per_model"] for c in comp]).mean().round(4).rename("mean score").to_frame())

print("\nmean pairwise scores")
pairs = pd.DataFrame([{"pair": " <-> ".join(p["pair"]), **{k: p[k] for k in ("cer_raw", "cer_norm", "wer_norm")}}
                      for c in comp for p in c["pairs"]])
display(pairs.groupby("pair").mean().round(4))
print(f"\nunanimous (all three agree after normalization): {100 * np.mean([c['unanimous'] for c in comp]):.1f}% of segments")

print("\nexample lines")
for r in res[:3]: print(json.dumps(r, ensure_ascii=False)[:400])

Expected output:

```
(fill in from a run)
```

# Part 3. Speaker diarization

Who spoke when: pyannote's Community-1 pipeline on the whole recording, then, in 3.5, a speaker for every segment of
Part 1 next to its text from Part 2. It is the code of the manual's *Speaker Diarization* section; the manual explains
what each output means and where the model fails. The section's scoring and tuning steps need a labelled reference,
so they are in *Part 3 extras*, on the manual's example minute.

## 3.1 Load the recording

Set `DIAR_REFERENCE` to an RTTM of who spoke when, if you have one: 3.3 then plots it above the output.

In [ ]:
# 3.1 load the recording, and its reference if you have one
import soundfile as sf
from IPython.display import Audio, display

DIAR_REFERENCE = None  # an RTTM of who spoke when, e.g. "data/lotusdis-S075/S075_300-360.reference.rttm"
wav_name = os.path.basename(AUDIO_PATH)

x, sr = sf.read(AUDIO_PATH, dtype="float32", always_2d=True)
waveform = torch.tensor(x.T)  # pyannote wants (channel, time)
duration = waveform.shape[1] / sr
print(wav_name, round(duration, 1), "s,", sr, "Hz,", waveform.shape[0], "channel")
display(Audio(x.mean(1), rate=sr))

Expected output:

```
(fill in from a run)
```

## 3.2 Run Community-1

In [ ]:
# 3.2 run Community-1 on the whole recording
import time
from pyannote.audio import Pipeline
from pyannote.audio.core.task import Problem, Resolution, Specifications

assert HF_TOKEN, "no HF_TOKEN: see the top of the notebook"
# torch >= 2.6 loads only the classes it is told are safe; these four are pyannote's own, stored in its checkpoints
torch.serialization.add_safe_globals([Specifications, Problem, Resolution, torch.torch_version.TorchVersion])

pipe = Pipeline.from_pretrained("pyannote/speaker-diarization-community-1", token=HF_TOKEN)
if pipe is None:
    raise SystemExit("accept the model's conditions on huggingface.co first")
pipe.to(torch.device(DEVICE))
t0 = time.time()
out = pipe({"waveform": waveform, "sample_rate": sr})
pipelines, outputs = {"Community-1": pipe}, {"Community-1": out.speaker_diarization}
print(f"{time.time() - t0:.1f} s,", len(out.speaker_diarization.labels()), "speakers,",
      len(list(out.speaker_diarization)), "turns,", len(list(out.exclusive_speaker_diarization)), "exclusive turns")

Expected output:

```
(fill in from a run)
```

The time depends on the machine: a T4 is many times faster than a Mac CPU. The exclusive output keeps one speaker at a
time; 3.5 uses it.

- `accept the model's conditions`, or `401 Unauthorized`: see the manual's Troubleshooting.
- On a GPU it also prints a `ReproducibilityWarning`: pyannote turns TF32 off. Leave it off.
- On a Mac with FFmpeg 9 (Homebrew) it also printed `torchcodec is not installed correctly` (torchcodec 0.9 supports
  FFmpeg 4 to 8) and a `UserWarning` about `std()`. This notebook passes the audio in memory, so the run finished normally.
- **Do not print the pipeline object:** on 2026-10-28 that printed the Hugging Face token, which was then revoked.

## 3.3 Look and listen

In [ ]:
# 3.3 plot the reference (if there is one) above the output
import matplotlib.pyplot as plt
from pyannote.core import Annotation, Segment, notebook

rows = list(outputs.items())
if DIAR_REFERENCE:
    ref = Annotation(uri=wav_name)
    for i, line in enumerate(open(DIAR_REFERENCE)):
        f = line.split()
        ref[Segment(float(f[3]), float(f[3]) + float(f[4])), i] = f[7]  # one track per line, or overlaps get merged
    rows = [("reference", ref)] + rows

notebook.crop = Segment(0, duration)
fig, axes = plt.subplots(len(rows), 1, figsize=(12, 2 * len(rows)), sharex=True, squeeze=False)
for ax, (title, ann) in zip(axes[:, 0], rows):
    notebook.plot_annotation(ann, ax=ax, time=True, legend=True)
    ax.set_title(title, loc="right")
plt.tight_layout()
plt.show()

Expected: one row, Community-1's speakers over time (two rows with a reference). Play the recording under 3.1 and
listen where the colours change.

**Give each RTTM line its own track,** as this step does with `i`. In pyannote, two turns over exactly the same stretch
overwrite each other otherwise.

## 3.4 Save the result

In [ ]:
# 3.4 save the result as RTTM next to the audio, and download it in Colab
diar.uri = wav_name[:-4]  # the file field of every line; without it pyannote writes <NA>
rttm_out = os.path.join(AUDIO_DIR, wav_name[:-4] + ".community-1.rttm")
with open(rttm_out, "w") as f:
    diar.write_rttm(f)
print(*open(rttm_out).read().splitlines()[:3], sep="\n")
if IN_COLAB:
    from google.colab import files
    files.download(rttm_out)

Expected output:

```
(fill in from a run)
```

One turn per line: field 2 is the file, 4 the start and 5 the length in seconds , and 8 the label.

## 3.5 Who said what

Joins the three parts: every segment of Part 1, with its time, the speaker with the most time inside it (from the
exclusive turns of 3.2) and its text from Part 2's `results.jsonl`. `speaker_share` is that speaker's part of the
segment's speech. Below 0.8 the segment may span a change of speaker: listen to it. Saved as
`<name>.who-said-what.csv`, with a byte-order mark so that Excel shows the Thai.

In [ ]:
# 3.5 a speaker and a text for every segment
import pandas as pd

MIXED_BELOW = 0.8  # flag segments whose main speaker has less than this share of the speech

exclusive = [(seg.start, seg.end, label) for seg, _, label in out.exclusive_speaker_diarization.itertracks(yield_label=True)]
texts = {r["audio_path"]: r["text"] for r in read_jsonl(RESULTS_JSONL)}
who = []
for name, (s, e) in SEG_TIMES.items():
    talk = {}
    for ts, te, label in exclusive:
        overlap = min(e, te) - max(s, ts)
        if overlap > 0:
            talk[label] = talk.get(label, 0) + overlap
    speaker = max(talk, key=talk.get) if talk else None
    who.append({"segment": name, "start_sec": round(s, 3), "end_sec": round(e, 3), "speaker": speaker,
                "speaker_share": round(talk[speaker] / sum(talk.values()), 3) if speaker else None,
                "text": texts.get(name)})
who = pd.DataFrame(who)

who_csv = os.path.join(AUDIO_DIR, wav_name[:-4] + ".who-said-what.csv")
who.to_csv(who_csv, index=False, encoding="utf-8-sig")
mixed = who.speaker_share.lt(MIXED_BELOW).sum()
print(f"{len(who)} segments: {who.speaker.isna().sum()} without a speaker, {who.text.isna().sum()} without a text,"
      f" {mixed} under {MIXED_BELOW:.0%} one speaker")
print(who.speaker.value_counts().to_string())
print("saved", who_csv)
display(who.head(10))

Expected output:

```
(fill in from a run)
```

A segment without a speaker lies where Community-1 heard nobody; one without a text is missing from `results.jsonl`.
The same table is what the `app/` page shows.

# Part 3 extras: score and tune on the example minute

The diarization notebook's Steps 6 and 8 to 10, unchanged except for names: they need a reference of who spoke when,
so they run on its example minute, `data/lotusdis-S075/` (LOTUSDIS, CC BY-SA 4.0; see the README), with the pipeline
of 3.2. Skip them for your own recording. The expected outputs are the diarization notebook's (Colab T4, 2026-11-11);
a Mac run on CPU gave the same numbers.

## E.1 Load and diarize the minute

In [ ]:
# E.1 the example minute, its corpus reference, and Community-1 on it
S075_PATH = "data/lotusdis-S075/S075_300-360.wav"
S075_DIR = os.path.dirname(S075_PATH)
s075_name = os.path.basename(S075_PATH)

x75, sr75 = sf.read(S075_PATH, dtype="float32", always_2d=True)
waveform75 = torch.tensor(x75.T)
duration75 = waveform75.shape[1] / sr75

ref = Annotation(uri=s075_name)
for i, line in enumerate(open(os.path.join(S075_DIR, "S075_300-360.reference.rttm"))):
    f = line.split()
    ref[Segment(float(f[3]), float(f[3]) + float(f[4])), i] = f[7]  # one track per line, or overlaps get merged

out75 = pipe({"waveform": waveform75, "sample_rate": sr75})
outputs75 = {"Community-1": out75.speaker_diarization}
print(s075_name, round(duration75, 1), "s,", len(out75.speaker_diarization.labels()), "speakers,",
      len(list(out75.speaker_diarization)), "turns,", len(list(out75.exclusive_speaker_diarization)), "exclusive turns")

Expected output:

```
S075_300-360.wav 60.0 s, 3 speakers, 31 turns, 26 exclusive turns
```

Three labels, as many as the meeting has people. The corpus reference names them M35, M36 and M37.

## E.2 See what Voxdiar would send Voxpick

Downloads Voxdiar's own overlap code, pinned to one Voxglobal commit, then runs it on the minute and on two short clips
cut around one overlap, each diarized on its own as Voxdiar does.

In [ ]:
# E.2 what Voxdiar would send Voxpick, for the minute and for two short clips (Voxglobal commit 429e148)
import urllib.request

OVERLAP_URL = ("https://raw.githubusercontent.com/thirawarit/voxglobal/"
               "429e1488fae1b60dcf9605c3d300a938ba080375/voxdiar/src/voxdiar/overlap.py")
if not os.path.exists("overlap.py"):
    urllib.request.urlretrieve(OVERLAP_URL, "overlap.py")
from overlap import analyze_turns

def voxdiar(diar, seconds):
    turns = [(label, seg.start, seg.end) for seg, _, label in diar.itertracks(yield_label=True)]
    a = analyze_turns(turns, duration_seconds=seconds, min_overlap_seconds=0.1)  # voxdiar/configs/default.yaml
    return (f"speakerCount {a.speaker_count}, overlap {a.overlap_seconds:.2f} s, overlapRatio {a.overlap_ratio:.4f}"
            f" | Multiple speakers: {a.speaker_count > 1}, Overlapping speech: {a.overlap_ratio > 0.02}")

diar = outputs75["Community-1"]
print("the minute:", voxdiar(diar, duration75))
for start, end in [(52.0, 55.0), (51.5, 55.5)]:
    clip = waveform75[:, int(start * sr75):int(end * sr75)]
    clip_out = pipe({"waveform": clip, "sample_rate": sr75}).speaker_diarization
    print(f"clip {start}-{end} s:", voxdiar(clip_out, clip.shape[1] / sr75))

Expected output:

```
the minute: speakerCount 3, overlap 0.91 s, overlapRatio 0.0152 | Multiple speakers: True, Overlapping speech: False
clip 52.0-55.0 s: speakerCount 1, overlap 0.00 s, overlapRatio 0.0000 | Multiple speakers: False, Overlapping speech: False
clip 51.5-55.5 s: speakerCount 2, overlap 0.19 s, overlapRatio 0.0464 | Multiple speakers: True, Overlapping speech: True
```

The two thresholds are Voxpick's defaults (`voxpick/src/types.ts`). The 3 s clip holds 1.50 s of overlap by our hand
labels, yet comes back with one speaker and no tag; cut half a second wider, it gets both tags. On a short clip, a tag
can depend on where the clip was cut: listen before you trust it.

## E.3 Against the corpus reference

In [ ]:
# E.3 score against the corpus reference (collar 0, overlap counted)
from pyannote.core import Timeline
from pyannote.metrics.diarization import DiarizationErrorRate

def score(reference, name):
    for short, hyp in outputs75.items():
        metric = DiarizationErrorRate(collar=0.0, skip_overlap=False)
        d = metric(reference, hyp, uem=Timeline([Segment(0, duration75)]), detailed=True)
        total = d["total"]
        print(f"{short} vs {name}: DER {d['diarization error rate']:.2%}, false alarm {d['false alarm'] / total:.2%},"
              f" missed {d['missed detection'] / total:.2%}, confusion {d['confusion'] / total:.2%},"
              f" reference speech {total:.1f} s")
        print("   labels matched:", metric.optimal_mapping(reference, hyp))

score(ref, "the corpus")

Expected output:

```
Community-1 vs the corpus: DER 43.72%, false alarm 0.00%, missed 42.11%, confusion 1.60%, reference speech 91.5 s
   labels matched: {'SPEAKER_00': 'M36', 'SPEAKER_01': 'M37', 'SPEAKER_02': 'M35'}
```

Almost all of the error is missed speech: time the reference gives to a second or third person. The minute has 91.5 s of reference speech because overlap counts twice.

## E.4 Against your own labels

Label the whole recording in Audacity and export the label track as a text file (start, end and name, separated by
tabs); time outside your labels counts as nobody talking. The manual's *Score it against a reference* shows how, and why
the labelling rules change the score. Here, our hand labels for the minute.

In [ ]:
# E.4 score against your own Audacity labels
LABELS_PATH = os.path.join(S075_DIR, "S075_300-360.hand-labels.txt")  # your Audacity export
mine = Annotation(uri="labels")
for i, line in enumerate(open(LABELS_PATH)):
    if line.startswith("\\"):  # Audacity adds these for spectral selections
        continue
    start, end, speaker = line.rstrip("\n").split("\t")
    mine[Segment(float(start), float(end)), i] = speaker

score(mine, "your labels")

Expected output:

```
Community-1 vs your labels: DER 20.94%, false alarm 0.10%, missed 18.40%, confusion 2.45%, reference speech 64.8 s
   labels matched: {'SPEAKER_00': 'Spk_A', 'SPEAKER_01': 'Spk_B', 'SPEAKER_02': 'Spk_C'}
```

Missed speech falls from about 42% to about 18% on the same output. **Score against labels whose rules you know.**

## E.5 Tuning: Settings, speaker counts, one clustering setting

Prints the pipeline's settings, runs three speaker counts and one changed clustering setting, then puts the defaults
back. The manual's *Tuning* section says what each setting acts on.

In [ ]:
# E.5 Tuning: the pipeline's settings, then speaker counts, then one clustering setting, then the defaults again
import warnings

def speakers(pipe, **count):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        labels = pipe({"waveform": waveform75, "sample_rate": sr75}, **count).speaker_diarization.labels()
    note = " (warned: outside the bounds)" if any("outside" in str(w.message) for w in caught) else ""
    return f"{len(labels)} speakers{note}"

for short, pipe in pipelines.items():
    defaults = pipe.parameters(instantiated=True)
    print(short)
    print("  parameters:", defaults)
    print("  segmentation_step:", pipe.segmentation_step, "| embedding_exclude_overlap:", pipe.embedding_exclude_overlap,
          "| batch sizes (segmentation, embedding):", pipe.segmentation_batch_size, pipe.embedding_batch_size)
    print("  default:            ", speakers(pipe))
    print("  num_speakers=2:     ", speakers(pipe, num_speakers=2))
    print("  min_speakers=4:     ", speakers(pipe, min_speakers=4))
    change = {"clustering": {"threshold": 0.8}}
    pipe.instantiate(change)
    print(f"  {change}:", speakers(pipe))
    pipe.instantiate(defaults)  # back to the defaults, so later runs use them
    print("  defaults again:     ", speakers(pipe))

Expected output:

```
Community-1
  parameters: {'segmentation': {'min_duration_off': 0.0}, 'clustering': {'threshold': 0.6, 'Fa': 0.07, 'Fb': 0.8}}
  segmentation_step: 0.1 | embedding_exclude_overlap: True | batch sizes (segmentation, embedding): 32 32
  default:             3 speakers
  num_speakers=2:      2 speakers
  min_speakers=4:      4 speakers
  {'clustering': {'threshold': 0.8}}: 3 speakers
  defaults again:      3 speakers
```

`min_speakers=4` gave four labels on a three-speaker minute, with no warning. **Pass a count only when you know it.**